> 在线讲义：[第 06 章 · 需求如何推动架构：TPU、SmartNIC、Unified Bus](https://wikieden.github.io/ai-infra-from-constraints/lessons/06-architecture.html) · 本书《从约束推导 AI Infra》

# 第 6 章 · 需求如何推动架构：TPU、SmartNIC、Unified Bus

配套讲义：仓库里 `ai-infra/notes/lessons/06-architecture.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 回本点（编的数） | 投入 ÷ 每次节省 | `breakeven(5e7, 0.01 / 1000)` | $5 \times 10^{12}$ 次 |
| 第二步 要多少天 | 回本数 ÷ 每天请求 | `days_to_breakeven(5e7, 0.01 / 1000, 1e10)` | 500 天；每天 $10^{11}$ 次是 50 天 |
| 第三步 40 Gbit/s 的包率 | $40 \times 10^9 / 672$ | `packet_rate(40)` | 约 5950 万包/秒 |
| 第三步 要几个核 | $\lambda / \mu$ | `cores(packet_rate(40), 2e7)`、`cores(…, 1e7)` | 3 到 6 个 |

40 Gbit/s、最小帧 84 字节、单核 1000 万–2000 万包/秒取自 AII 第 1 章 §1.4.2 与脚注；
专用芯片回本的 5000 万元、每 1000 次省 0.01 元、每天 1e10 次请求是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math

**函数 `breakeven`**：回本请求数 = 前期投入 ÷ 每次请求省下的钱。

In [ ]:
def breakeven(upfront: float, saving_per_request: float) -> float:
    """回本请求数 = 前期投入 ÷ 每次请求省下的钱。"""
    return upfront / saving_per_request

**函数 `days_to_breakeven`**：按每天的请求量，多少天回本。

In [ ]:
def days_to_breakeven(upfront: float, saving_per_request: float, requests_per_day: float) -> float:
    """按每天的请求量，多少天回本。"""
    return breakeven(upfront, saving_per_request) / requests_per_day

**函数 `packet_rate`**：最小帧条件下每秒到达多少个包：比特率 ÷ (每包线路字节 × 8)。

In [ ]:
def packet_rate(gbps: float, wire_bytes: int = 84) -> float:
    """最小帧条件下每秒到达多少个包：比特率 ÷ (每包线路字节 × 8)。"""
    return gbps * 1e9 / (wire_bytes * 8)

**函数 `cores`**：光转发要几个核：λ / μ，向上取整。

In [ ]:
def cores(rate: float, per_core: float) -> int:
    """光转发要几个核：λ / μ，向上取整。"""
    return math.ceil(rate / per_core)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步：专用芯片什么时候回本（编的数）")
n = breakeven(5e7, 0.01 / 1000)
print(f"  5000 万元 ÷ 每次省 1e-5 元 = {n:.0e} 次请求")
for per_day in (1e10, 1e11):
    print(f"  每天 {per_day:.0e} 次 → {days_to_breakeven(5e7, 0.01 / 1000, per_day):.0f} 天")

In [ ]:
print("\n第三步：40 Gbit/s、最小帧 84 字节")
lam = packet_rate(40)
print(f"  包率 = 40e9 ÷ 672 ≈ {lam / 1e6:.1f} 百万包/秒")
print(f"  单核 2000 万包/秒 → {cores(lam, 2e7)} 个核；单核 1000 万包/秒 → {cores(lam, 1e7)} 个核")

## 3. 代码里才看得到的

1. 84 字节不是以太网帧本身（最小 64 字节），而是加上前导码和帧间隔以后在线路上占的长度；用 64 字节算，包率会高估约 30%。
2. `cores` 向上取整：5.95 个核实际要 6 个；原文写"3–6"是按单核 2000 万和 1000 万两头算的（2.98 → 3，5.95 → 6）。
3. 回本的例子里，"每 1000 次省 0.01 元"要先换成每次省 $10^{-5}$ 元再除，单位不统一很容易差三个数量级。

## 4. 自己改一改

- 把 `packet_rate` 的帧长从 84 改成 1538（最大帧在线路上的长度），看要几个核。
- 把回本例子的前期投入翻倍，看每天 $10^{10}$ 次请求要多少天。
- 把单核处理能力提高到 5000 万包/秒，40 Gbit/s 还要几个核？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 比例一变，分工重排；
> 需求翻倍造专芯，回本看规模；
> 网速涨了核不够，包处理交给网卡；
> 卡多了先算容量，再算读写，最后看谁在等；
> 老办法新比例，数据搬移定架构。